In [30]:
from imports.models import *
from imports.utils import *
import adabound as Adabound
import itertools

In [9]:
from multiprocessing import Pool
POOL_PROCESS = 10

In [6]:
# time index dictionary file corresponding target periods (see Supplementary Note 1)
total_indices = np.load('../tutorial_data/total_indices_v4_full.npy', allow_pickle=True).item()

# (A, B) pairs which corresponds each target period. (Equation is described in the paper)
AB_lists_dic = np.load('../tutorial_data/AB_target_dic_v4_s0.npy', allow_pickle=True).item()
for i in range(1, 16):
    temp = np.load('../tutorial_data/AB_target_dic_v4_s{}.npy'.format(i), allow_pickle=True).item()
    AB_lists_dic.update(temp)

In [11]:
# 파라미터 값
A_SIDE_NUM = 3
A_SIDE_RESOL = 600
A_TARGET_MARGIN = 25
A_SIDE_MARGIN = 300
A_FAR_SIDE_MARGIN = 5000
A_HIER_MARGIN = 25
A_HIER_MARGIN = 25
SIDE_CANDI_NUM = 2
B_TARGET_GAP = 0

In [12]:
PRE_PROCESS = False
PRE_SCALE = 1
MAGNETIC_FIELD = 403.553                        # # The external magnetic field strength. Unit: Gauss
GYRO_MAGNETIC_RATIO = 1.0705*1000               # Unit: Herts
WL_VALUE = MAGNETIC_FIELD*GYRO_MAGNETIC_RATIO*2*np.pi

In [26]:

class Regression_Model():

    def __init__(self, *args):

        self.CUDA_DEVICE, self.N_PULSE, self.IMAGE_WIDTH, self.TIME_RANGE_32, self.TIME_RANGE_256, self.EXISTING_SPINS, \
            self.A_init, self.A_final, self.A_step, self.A_range, self.B_init, self.B_final, self.zero_scale, \
                self.noise_scale, self.SAVE_DIR_NAME, self.model_lists, self.target_side_distance, self.is_CNN = args

        self.exp_data_32 = np.load('../data/exp_data/exp_data_32.npy')
        self.exp_data_deno_32 = np.load('../data/exp_data/exp_data_32_deno.npy')
        self.exp_data_256 = np.load('../data/exp_data/exp_data_256.npy')
        self.exp_data_deno_256 = np.load('../data/exp_data/exp_data_256_deno.npy')
        self.time_data_32 = np.load('../data/time_data/time_data_32.npy')
        self.time_data_256 = np.load('../data/time_data/time_data_256.npy')
        self.spin_bath_32 = np.load('../data/spin_bath/spin_bath_M_value_N32.npy')
        self.spin_bath_256 = np.load('../data/spin_bath/spin_bath_M_value_N256.npy')
        self.total_indices_32 = np.load('../data/total_indices/total_indices_v4_N32.npy', allow_pickle=True).item()
        self.total_indices_256 = np.load('../data/total_indices/total_indices_v4_N256.npy', allow_pickle=True).item()

        if self.EXISTING_SPINS:
            deno_pred_N32_B15000_above = np.load('../data/results/deno_pred_N32_B15000_above.npy')

        self.pool = Pool(processes=POOL_PROCESS)

    def close_pool(self):
        self.pool.close()
        self.pool.terminate()
        self.pool.join()

    def estimate_specific_AB_values(self, predicted_periods):

        model_lists = np.array([[A_temp_list[0], A_temp_list[-1], self.B_init, self.B_final] for A_temp_list in predicted_periods])

        total_raw_pred_list = []
        total_deno_pred_list = []
        total_A_lists = []
        total_results = []
        print("==========================================================================================")
        print('image_width:{}, TIME_RANGE_32:{}, TIME_RANGE_256:{}, B_init:{}, B_end:{}'.format(self.IMAGE_WIDTH, self.TIME_RANGE_32, self.TIME_RANGE_256, self.B_init, self.B_final))
        print("==========================================================================================")

        for model_idx, [A_first, A_end, B_first, B_end] in enumerate(model_lists):
            print("========================================================================")
            print('A_first:{}, A_end:{}, B_first:{}, B_end:{}'.format(A_first, A_end, B_first, B_end))
            print("========================================================================")
            A_num = 1
            B_num = 1
            A_resol, B_resol = 50, B_end-B_first

            A_idx_list = np.arange(A_first, A_end+A_resol, A_num*A_resol)
            if (B_end-B_first)%B_resol==0:
                B_idx_list = np.arange(B_first, B_first+B_resol, B_num*B_resol)
            else:
                B_idx_list = np.arange(B_first, B_end, B_num*B_resol)
            AB_idx_set = [[A_idx, B_idx] for A_idx, B_idx in itertools.product(A_idx_list, B_idx_list)]

            A_side_num = A_SIDE_NUM
            A_side_resol = A_SIDE_RESOL
            A_target_margin = A_TARGET_MARGIN
            A_side_margin = A_SIDE_MARGIN
            A_far_side_margin = A_FAR_SIDE_MARGIN
            A_hier_margin = A_HIER_MARGIN
            side_candi_num = SIDE_CANDI_NUM        # the number of "how many times" to generate 'AB_side_candidate'

            if self.N_PULSE==32:
                B_side_min, B_side_max = 6000, 70000
                B_side_gap = 5000
                B_target_gap = 1000  # distance between targets only valid when B_num >= 2.
                distance_btw_target_side = A_target_margin+A_side_margin+self.target_side_distance # the final distance between target and side = distance_btw_target_side - (A_target_margin+A_side_margin)

            elif self.N_PULSE==256:
                B_side_min, B_side_max = 1000, 20000
                B_side_gap = 0      # distance between target and side (applied for both side_same and side)
                B_target_gap = 0
                distance_btw_target_side = A_target_margin+A_side_margin+self.target_side_distance

            PRE_PROCESS, PRE_SCALE = False, 1
            if ((self.N_PULSE == 32) & (B_first<12000)):
                PRE_PROCESS = True
                PRE_SCALE = 8
                print("==================== PRE_PROCESSING:True =====================")

            class_num = A_num*B_num + 1
            cpu_num_for_multi = 10
            batch_for_multi = 256
            class_batch = cpu_num_for_multi*batch_for_multi

            spin_zero_scale = {'same':self.zero_scale, 'side':0.50, 'mid':0.05, 'far':0.05}  # setting 'same'=1.0 for hierarchical model

            epochs = 20
            valid_batch = 4096
            valid_mini_batch = 1024
            num_of_summation = 4

            args = (AB_lists_dic, self.N_PULSE, A_num, B_num, A_resol, B_resol, A_side_num, A_side_resol, B_side_min,
                        B_side_max, B_target_gap, B_side_gap, A_target_margin, A_side_margin, A_far_side_margin,
                        class_batch, class_num, spin_zero_scale, distance_btw_target_side, side_candi_num)

            for class_idx in range(num_of_summation):
                TPk_AB_candi, _, temp_hier_target_AB_candi  = gen_TPk_AB_candidates(AB_idx_set, True, *args)
                temp_hier_target_AB_candi[:,:,0] = get_marginal_arr(temp_hier_target_AB_candi[:,:,0], A_hier_margin)
                if class_idx==0:
                    total_hier_target_AB_candi = temp_hier_target_AB_candi[:]
                    target_candidates = TPk_AB_candi[1, :, 0, :]
                    side_candidates   = TPk_AB_candi[0, :, 0, :]
                    rest_candidates   = TPk_AB_candi[1, :, 1:, :]
                else:
                    total_hier_target_AB_candi = np.concatenate((total_hier_target_AB_candi, temp_hier_target_AB_candi[:]), axis=1)
                    target_candidates = np.concatenate((target_candidates, TPk_AB_candi[1, :, 0, :]), axis=0)
                    side_candidates   = np.concatenate((side_candidates, TPk_AB_candi[0, :, 0, :]), axis=0)
                    rest_candidates   = np.concatenate((rest_candidates, TPk_AB_candi[1, :, 1:, :]), axis=0)

            hier_indices = return_total_hier_index_list(A_idx_list, cut_threshold=4)
            if len(hier_indices)==0 or len(hier_indices[-1])==0 or len(hier_indices[-1][0])==0:
                print("⚠️ hier_indices가 비어 있거나 구조적으로 잘못되었습니다. total_class_num = 1")
            total_class_num = len(hier_indices[-1][0]) + 1

            total_TPk_AB_candidates = np.zeros((total_class_num, num_of_summation*TPk_AB_candi.shape[1], total_class_num+TPk_AB_candi.shape[2]+2, 2))
            indices = np.random.randint(rest_candidates.shape[0], size=(total_class_num, rest_candidates.shape[0]))
            total_TPk_AB_candidates[:, :, (total_class_num-1):-4, :] = rest_candidates[indices]
            indices = np.random.randint(side_candidates.shape[0], size=(total_class_num, side_candidates.shape[0], 2))
            total_TPk_AB_candidates[:, :, -4:-2, :] = side_candidates[indices]
            indices = np.random.randint(side_candidates.shape[0], size=(total_class_num, side_candidates.shape[0], 2))
            total_TPk_AB_candidates[:, :, -2:, :] = side_candidates[indices]

            # if self.EXISTING_SPINS:
            #     total_TPk_AB_candidates = return_existing_spins_wrt_margins(deno_pred_N32_B15000_above, total_TPk_AB_candidates, A_existing_margin, B_existing_margin)
            final_TPk_AB_candidates = total_TPk_AB_candidates[:1]
            for class_idx, hier_index in enumerate(hier_indices):
                temp_batch = total_TPk_AB_candidates.shape[1] // len(hier_index)
                for idx2, index in enumerate(hier_index):
                    temp = np.swapaxes(total_hier_target_AB_candi[index], 0, 1)
                    if idx2 < (len(hier_index)-1):
                        temp_idx = np.random.randint(total_hier_target_AB_candi.shape[1], size=(temp_batch))
                        total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:(idx2+1)*temp_batch, :len(index)] = temp[temp_idx]
                    else:
                        residual_batch = total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:, :len(index)].shape[0]
                        temp_idx = np.random.randint(total_hier_target_AB_candi.shape[1], size=(residual_batch))
                        total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:, :len(index)] = temp[temp_idx]
                final_TPk_AB_candidates = np.concatenate((final_TPk_AB_candidates, total_TPk_AB_candidates[class_idx+1:class_idx+2, :, :]), axis=0)
            median_A_idx = len(AB_idx_set) // 2

            # below is for generating N32 datasets
            print("Generating N32 data..")
            model_index_32 = get_model_index(self.total_indices_32, AB_idx_set[median_A_idx][0], time_thres_idx=self.TIME_RANGE_32, image_width=self.IMAGE_WIDTH)
            if (AB_idx_set[median_A_idx][0] > 13000) | (AB_idx_set[median_A_idx][0] < -13000):
                model_index_32, _ = return_index_without_A_idx(self.total_indices_32, model_index_32, 0, self.TIME_RANGE_32, 5)
            total_class_num = final_TPk_AB_candidates.shape[0]
            X_train_arr = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], model_index_32.shape[0], 2*self.IMAGE_WIDTH+1))
            Y_train_arr = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], total_class_num))
            mini_batch = X_train_arr.shape[1] // cpu_num_for_multi
            for class_idx in range(total_class_num):
                Y_train_arr[class_idx, :, class_idx] = 1
                for idx1 in range(cpu_num_for_multi):
                    AB_lists_batch = final_TPk_AB_candidates[class_idx, idx1*mini_batch:(idx1+1)*mini_batch]
                    globals()["pool_{}".format(idx1)] = self.pool.apply_async(gen_M_arr_batch, [AB_lists_batch, model_index_32, self.time_data_32[:self.TIME_RANGE_32],
                                                                                            WL_VALUE, 32, PRE_PROCESS, PRE_SCALE,
                                                                                            self.noise_scale, self.spin_bath_32[:self.TIME_RANGE_32]])

                for idx2 in range(cpu_num_for_multi):
                    X_train_arr[class_idx, idx2*mini_batch:(idx2+1)*mini_batch] = globals()["pool_{}".format(idx2)].get(timeout=None)
                print("class_idx:", class_idx, end=' ')

            X_train_arr = X_train_arr.reshape(-1, model_index_32.flatten().shape[0])
            Y_train_arr = Y_train_arr.reshape(-1, Y_train_arr.shape[2])

            # below is for generating N256 datasets
            if self.TIME_RANGE_256:
                print("Generating N256 data..")
                model_index_256 = get_model_index(self.total_indices_256, AB_idx_set[median_A_idx][0], time_thres_idx=self.TIME_RANGE_256, image_width=self.IMAGE_WIDTH)
                if (AB_idx_set[median_A_idx][0] > 13000) | (AB_idx_set[median_A_idx][0] < -13000):
                    model_index_256, _ = return_index_without_A_idx(self.total_indices_256, model_index_256, 0, self.TIME_RANGE_256, 5)
                total_class_num = final_TPk_AB_candidates.shape[0]
                X_train_256 = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], model_index_256.shape[0], 2*self.IMAGE_WIDTH+1))
                Y_train_256 = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], total_class_num))
                mini_batch = X_train_256.shape[1] // cpu_num_for_multi
                for class_idx in range(total_class_num):
                    Y_train_256[class_idx, :, class_idx] = 1
                    for idx1 in range(cpu_num_for_multi):
                        AB_lists_batch = final_TPk_AB_candidates[class_idx, idx1*mini_batch:(idx1+1)*mini_batch]
                        globals()["pool_{}".format(idx1)] = self.pool.apply_async(gen_M_arr_batch, [AB_lists_batch, model_index_256, self.time_data_256[:self.TIME_RANGE_256],
                                                                                                WL_VALUE, 256, PRE_PROCESS, PRE_SCALE,
                                                                                                self.noise_scale, self.spin_bath[:self.TIME_RANGE_256]])

                    for idx2 in range(cpu_num_for_multi):
                        X_train_256[class_idx, idx2*mini_batch:(idx2+1)*mini_batch] = globals()["pool_{}".format(idx2)].get(timeout=None)
                    print("class_idx:", class_idx, end=' ')

                X_train_256 = X_train_256.reshape(-1, model_index_256.flatten().shape[0])
                Y_train_256 = Y_train_256.reshape(-1, Y_train_256.shape[2])
                X_train_arr = np.concatenate((X_train_32, X_train_256), axis=1)

            model = HPC(X_train_arr.shape[-1], Y_train_arr.shape[-1]).cuda()
            try:
                model(torch.Tensor(X_train_arr[:16]).cuda())
            except:
                raise NameError("The input shape should be revised")

            total_parameter = sum(p.numel() for p in model.parameters())
            print('total_parameter: ', total_parameter / 1000000, 'M')

            MODEL_PATH = './data/models/'
            mini_batch_list = [2048]
            learning_rate_list = [1e-6]
            op_list = [['Adabound', [50, 25, 10, 5]]]
            criterion = nn.BCELoss().cuda()

            hyperparameter_set = [[mini_batch, learning_rate, selected_optim_name] for mini_batch, learning_rate, selected_optim_name in itertools.product(mini_batch_list, learning_rate_list, op_list)]
            print("==================== A_idx: {}, B_idx: {} ======================".format(A_first, B_first))

            total_loss, total_val_loss, total_acc, trained_model = train(MODEL_PATH, self.N_PULSE, X_train_arr, Y_train_arr, model, hyperparameter_set, criterion,
                                                                        epochs, valid_batch, valid_mini_batch, self.exp_data_32, is_pred=False, is_print_results=False,
                                                                        is_preprocess=PRE_PROCESS, PRE_SCALE=PRE_SCALE, model_index=model_index_32,
                                                                        exp_data_deno=self.exp_data_deno_32, is_regression=False)

            model.load_state_dict(torch.load(trained_model[0][0]))
            model.eval()
            print("Model loaded as evalutation mode. Model path:", trained_model[0][0])

            if self.TIME_RANGE_256:
                exp_data_test = np.hstack((self.exp_data_256[model_index_256.flatten()], self.exp_data_32[model_index_32.flatten()]))
            else:
                exp_data_test = self.exp_data_32[model_index_32.flatten()]
            exp_data_test = 1-(2*exp_data_test - 1)
            exp_data_test = exp_data_test.reshape(1, -1)
            exp_data_test = torch.Tensor(exp_data_test).cuda()

            pred = model(exp_data_test)
            pred = pred.detach().cpu().numpy()
            total_raw_pred_list.append([pred[0], total_class_num, hier_indices[-1][0].__len__()])
            print("raw", np.argmax(pred), np.max(pred), pred)

            if self.TIME_RANGE_256:
                exp_data_test = np.hstack((self.exp_data_deno_256[model_index_256.flatten()], self.exp_data_deno_32[model_index_32.flatten()]))
            else:
                exp_data_test = self.exp_data_deno_32[model_index_32.flatten()]
            exp_data_test = 1-(2*exp_data_test - 1)
            exp_data_test = exp_data_test.reshape(1, -1)
            exp_data_test = torch.Tensor(exp_data_test).cuda()

            pred = model(exp_data_test)
            pred = pred.detach().cpu().numpy()
            total_deno_pred_list.append([pred[0], total_class_num, hier_indices[-1][0].__len__()])
            print("deno", np.argmax(pred), np.max(pred), pred)
            print("Config:N_PULSE{}, B_init{}, B_final{}, w{}_batch{}, zero{}".format(self.N_PULSE, self.B_init, self.B_final, self.IMAGE_WIDTH, batch_for_multi, self.zero_scale))
            total_results.append([total_raw_pred_list[model_idx], total_deno_pred_list[model_idx], hier_indices])

        return total_results


In [27]:
SAVE_DIR_NAME = '../data/results/'
predicted_periods = np.load(SAVE_DIR_NAME + 'predicted_periods.npy', allow_pickle=True)

In [28]:
CUDA_DEVICE = 0
N_PULSE = 32
IMAGE_WIDTH = 10
TIME_RANGE_32  = 7000
TIME_RANGE_256  = 0
EXISTING_SPINS = 0
EVALUATION_ALL = 0

target_side_distance = 3000
A_init  = -30000
A_final = 30000
A_step  = 200
A_range = 200
B_init  = 1500
B_final = 50000
noise_scale = 0.5
zero_scale = 0.05
is_CNN = 0
is_remove_model_index = 0

In [32]:
model_lists = get_AB_model_lists(A_init, A_final, A_step, A_range, B_init, B_final)
regression_args = (CUDA_DEVICE, N_PULSE, IMAGE_WIDTH, TIME_RANGE_32, TIME_RANGE_256, EXISTING_SPINS, A_init, A_final, A_step, A_range, B_init, B_final, zero_scale, noise_scale, SAVE_DIR_NAME, model_lists, target_side_distance, is_CNN)
regression_model = Regression_Model(*regression_args)
regression_results = regression_model.estimate_specific_AB_values(predicted_periods[:10])
regression_model.close_pool()

image_width:10, TIME_RANGE_32:7000, TIME_RANGE_256:0, B_init:1500, B_end:50000
A_first:-30000, A_end:-29900, B_first:1500, B_end:50000
==================== PRE_PROCESSING:True =====================
Generating N32 data..
class_idx: 0 class_idx: 1 total_parameter:  3.579394 M
==================== A_idx: -30000, B_idx: 1500 ======================
train_batch:  57344 valid_batch:  4096


 Training Start:  Wed May 28 18:12:54 2025
 mini_batch: 2048  | learning_rate:  1e-06  | selected_optim_name:  ['Adabound', [50, 25, 10, 5]]  |


C:\Users\KISTQUANTUM\Desktop\Deep_Learning_CPMG_Analysis\Deep_Learning_CPMG_Analysis\adabound.py:94: UserWarning:

This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\python_arg_parser.cpp:1691.)



Epoch:    1  | Loss = 0.59205 | Val_loss: 0.41022 | Accuracy: 85.06 % | time: 1.182(s) | 

RuntimeError: Parent directory ./data/models does not exist.

In [ ]:
# B 값이 15000 이상인 하이퍼핀 파라미터만 필터링해서 저장
filtered_results = [res for res in regression_results if res[1] > 15000]  # res = [A, B] 형태라고 가정
filtered_results = np.array(filtered_results)

save_path = SAVE_DIR_NAME + 'predicted_results_N32_B15000above.npy'
np.save(save_path, filtered_results)
print(f"✅ B > 15000인 결과를 {save_path} 에 저장했습니다.")